# Chest X-Ray Tiered Classifier — Colab Training Notebook

Single notebook that trains every model in the project using the layered architecture in `core/`, `application/`, `infrastructure/`.

**Workflow**:
1. Mount Google Drive (project lives at `MyDrive/xray/`).
2. `pip install -r requirements.txt` + `requirements-training.txt`.
3. Acquire Ark+ checkpoint (falls back to Swin-Base ImageNet).
4. Train Tier 1 (MobileNetV2) → Tier 2 (EfficientNetB4) → Tier 2 (Ark+).
5. Calibrate temperature scaling + conformal prediction.
6. Run the A1–A15 ablation matrix and rebuild `outputs/results/ablation.json`.
7. Sync outputs back to Drive.

**Reproducibility**: every cell logs to MLflow under `experiments/mlruns/` on Drive. After training, run `scripts/build_ablation_json.py` locally — it pulls real metrics straight out of MLflow.

## 0. Environment + Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/xray'
assert os.path.isdir(PROJECT_ROOT), (
    f'Expected the project at {PROJECT_ROOT}. Copy the repo to your Drive first.'
)
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
!ls -la

In [ ]:
# Optional: pull the latest commits from origin if the Drive copy is git-tracked
import subprocess
try:
    subprocess.check_call(['git', 'pull', '--ff-only'])
except Exception as exc:
    print('git pull skipped:', exc)

In [ ]:
# Install runtime + training deps (use pinned versions from the repo)
!pip install --quiet -r requirements.txt
!pip install --quiet -r requirements-training.txt

In [ ]:
# Make the repo importable from anywhere in the notebook kernel
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 1. Acquire Ark+ Checkpoint (or Swin-Base fallback)

The script tries the official Ark release URLs first, then HuggingFace mirrors, then falls back to a clean Swin-Base ImageNet checkpoint via timm. Whichever succeeds, the downstream Tier 2 Ark+ trainer will use it.

In [ ]:
!python scripts/download_ark_plus.py
!ls -la outputs/models/ | grep -i 'ark\|swin'

## 2. Dataset Sanity Check

Assumes `data/processed/{train,val,test,calibration}.csv` already exist. If not, run `scripts/preprocess.py` first (uncomment below).

In [ ]:
# Uncomment if you need to (re)generate the splits from NIH Data_Entry_2017.csv
# !python scripts/preprocess.py

import pandas as pd
for split in ['train', 'val', 'test']:
    p = f'data/processed/{split}.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'{split}: {len(df)} rows, classes:', dict(df['label'].value_counts()) if 'label' in df.columns else '(no label col)')
    else:
        print(f'{split}: MISSING ({p})')

## 3. Train Tier 1 — MobileNetV2

Uses `application.services.training_service.TrainingService` which wires Trainer + observers (MLflow, checkpoint, early-stopping, LR logger, carbon tracker).

In [ ]:
from application.dto.training_config_dto import TrainingConfigDTO
from application.services.training_service import TrainingService

tier1_cfg = TrainingConfigDTO(
    backbone='mobilenet_v2',
    num_classes=2,
    epochs=50,
    batch_size=32,
    lr_backbone=1e-4,
    lr_head=1e-3,
    early_stopping_patience=7,
    seed=42,
    mlflow_run_name='Tier1_MobileNetV2_Colab',
    output_weights_path='outputs/models/best_tier1_mobilenet.pth',
)
service = TrainingService()
tier1_result = service.train_model(tier1_cfg)
print('Tier 1 done:', tier1_result)

## 4. Train Tier 2 — EfficientNetB4

In [ ]:
tier2_eff_cfg = TrainingConfigDTO(
    backbone='efficientnet_b4',
    num_classes=2,
    epochs=50,
    batch_size=16,
    lr_backbone=5e-5,
    lr_head=5e-4,
    early_stopping_patience=7,
    seed=42,
    mlflow_run_name='Tier2_EfficientNetB4_Colab',
    output_weights_path='outputs/models/best_tier2_efficientnet.pth',
)
tier2_eff_result = service.train_model(tier2_eff_cfg)
print('Tier 2 EffNetB4 done:', tier2_eff_result)

## 5. Train Tier 2 — Ark+ Swin

Uses the official Ark checkpoint if `outputs/models/ark_plus_swin_base.pth` exists, otherwise the Swin-Base ImageNet fallback. The Tier2ArkPlus class wires `freeze_epochs` for gradual unfreezing — adjust below for your GPU budget.

In [ ]:
tier2_ark_cfg = TrainingConfigDTO(
    backbone='ark_plus',
    num_classes=2,
    epochs=40,
    batch_size=12,
    lr_backbone=2e-5,
    lr_head=2e-4,
    early_stopping_patience=8,
    seed=42,
    mlflow_run_name='Tier2_ArkPlus_Colab',
    output_weights_path='outputs/models/best_tier2_arkplus.pth',
)
tier2_ark_result = service.train_model(tier2_ark_cfg)
print('Tier 2 Ark+ done:', tier2_ark_result)

## 6. Calibration — Temperature Scaling + Conformal Prediction

In [ ]:
from application.services.calibration_service import CalibrationService
from core.uncertainty.calibration import ModelWithTemperature

calib = CalibrationService()
calib.fit_temperature(
    weights_path='outputs/models/best_tier2_efficientnet.pth',
    backbone='efficientnet_b4',
    calibration_csv='data/processed/calibration.csv',
    output_path='outputs/results/temperature.pt',
)

from core.uncertainty.conformal import ConformalPredictor
conformal = ConformalPredictor(alpha=0.05)
# Re-load Tier 2 model and calibration set, then calibrate q_hat
# (See scripts/calibrate_conformal.py for the full pipeline.)
!python -c 'from application.services.calibration_service import CalibrationService; CalibrationService().fit_conformal_qhat(weights_path="outputs/models/best_tier2_efficientnet.pth", backbone="efficientnet_b4", calibration_csv="data/processed/calibration.csv", output_path="outputs/results/q_hat.pt")'

## 7. Ablation Matrix (A1–A15)

Drives `AblationRunner` which iterates the configurations from PLAN.md §5.4 and persists results to MLflow.

In [ ]:
from application.orchestration.ablation_runner import AblationRunner
runner = AblationRunner()
runner.run_all(start='A1', end='A15', dry_run=False)

## 8. Regenerate `outputs/results/ablation.json` Honestly

Reads real MLflow metrics — rows with missing or back-filled metrics are marked `preliminary_placeholder` with `null` numbers.

In [ ]:
!python scripts/build_ablation_json.py
!python -c 'import json; data = json.load(open("outputs/results/ablation.json")); print(f"Real rows: {sum(1 for r in data if r[\"provenance\"] == \"mlflow_run\")}/{len(data)}")'

## 9. Statistical Tests (DeLong + Bootstrap)

In [ ]:
!python scripts/statistical_tests.py --output outputs/results/statistical_comparison.csv

## 10. ONNX + INT8 Export (Mobile-Friendly)

In [ ]:
!python scripts/export_onnx.py --model tier1 --weights outputs/models/best_tier1_mobilenet.pth
!python scripts/export_onnx.py --model tier2 --weights outputs/models/best_tier2_efficientnet.pth --quantize int8

## 11. Sanity Check + Drive Sync

Drive is mounted at the project root, so every `outputs/`, `experiments/mlruns/`, and `data/processed/` write already lands on Drive. This cell is just a confirmation.

In [ ]:
!ls -la outputs/models/
!ls -la outputs/results/
!du -sh experiments/mlruns/ outputs/ 2>/dev/null